# 08 Data-Method-Performance Pathways

This notebook handles Subagent 08's Fig. 7 data-source, method, and environmental-performance pathway analysis. Inputs are fixed to the new master data `data/NSFC正式增量采集_2014-2026_去重筛选最终结果.csv` (N=9222) and the keyword table `data/NSFC正式增量采集_37个关键词.csv`; outputs are limited to:

- `output/figures/Fig7_data_method_performance_sankey.svg`
- `output/figures/Fig7_data_method_performance_sankey.pdf`
- `output/figures/Fig7_data_method_performance_sankey.tiff`
- `output/figures/Fig7_data_method_performance_sankey.png`
- `output/tables/08_sankey_links.csv`
- `output/logs/08_sankey_text.md`

Constraints: visible figure text is in English; the notebook does not modify `data/` and does not write files owned by other subagents. Export cells are retained for a later Fig7/table/log worker rerun; do not execute export cells when only validating scope.


## Figure Design Contract

- Core conclusion: BAE-related NSFC projects in the new master table are organized into traceable data-source, method-family, and environmental-performance pathways using the controlled 37-term keyword table.
- Evidence chain: each record identifies data-source, method-family, and environmental-performance categories from `matched_method_terms`, `matched_object_terms`, `matched_performance_terms`, and `matched_keywords` in the new master table; these fields are generated from the controlled vocabulary in `data/NSFC正式增量采集_37个关键词.csv`. Multilabel records use fractional weighting to avoid duplicate inflation.
- Path subset: only records whose `matched_performance_terms` map to explicit environmental-performance categories enter the Sankey; excluded records are assigned reasons for no performance term or unmapped performance terms.
- Archetype: schematic-led composite using three node columns plus variable-width Bezier ribbons, avoiding dependence on an interactive Sankey.
- Export contract: SVG, PDF, TIFF, and PNG are exported consistently with Python/matplotlib; SVG/PDF keep editable text where possible; the links table provides source data.


In [ ]:
# ===== 1. Environment Setup and Paths =====
# This notebook uses only Python, pandas, numpy, matplotlib, and the standard library for data preparation and static figure export.
# Code searches upward from the current directory for the project root so it can run from scratch from the project root or code/ directory.
from pathlib import Path
from datetime import datetime
import math
import re
import textwrap
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib as mpl
from matplotlib import font_manager
import matplotlib.pyplot as plt

# Register and require Times New Roman; stop immediately if the font is unavailable instead of using a fallback.
TIMES_NEW_ROMAN_PATHS = [
    Path("/Library/Fonts/Times New Roman.ttf"),
    Path("/Library/Fonts/Times New Roman Bold.ttf"),
    Path("/Library/Fonts/Times New Roman Italic.ttf"),
    Path("/Library/Fonts/Times New Roman Bold Italic.ttf"),
    Path("/System/Library/Fonts/Supplemental/Times New Roman.ttf"),
    Path("/System/Library/Fonts/Supplemental/Times New Roman Bold.ttf"),
    Path("/System/Library/Fonts/Supplemental/Times New Roman Italic.ttf"),
    Path("/System/Library/Fonts/Supplemental/Times New Roman Bold Italic.ttf"),
]
for font_path in TIMES_NEW_ROMAN_PATHS:
    if font_path.exists():
        font_manager.fontManager.addfont(str(font_path))
try:
    font_manager.findfont("Times New Roman", fallback_to_default=False)
except ValueError as exc:
    raise RuntimeError("Times New Roman is required for figure export but was not found by matplotlib.") from exc

from matplotlib.path import Path as MplPath
from matplotlib.patches import PathPatch, Rectangle
from matplotlib import colors as mcolors

# Set common publication-figure parameters: white background, no extra borders, and editable SVG/PDF text where possible.
mpl.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman"],
    "font.sans-serif": ["Times New Roman"],
    "font.monospace": ["Times New Roman"],
    "svg.fonttype": "none",
    "pdf.fonttype": 42,
    "font.size": 7,
    "axes.spines.right": False,
    "axes.spines.top": False,
    "axes.linewidth": 0.8,
    "legend.frameon": False,
})

MAIN_DATA_FILENAME = "NSFC正式增量采集_2014-2026_去重筛选最终结果.csv"
KEYWORD_FILENAME = "NSFC正式增量采集_37个关键词.csv"
EXPECTED_RECORDS_TOTAL = 9222


def find_project_root(start: Path) -> Path:
    """Search upward from the current path for the project root containing both the new master data and keyword table."""
    start = start.resolve()
    for candidate in [start, *start.parents]:
        data_dir = candidate / "data"
        if (data_dir / MAIN_DATA_FILENAME).exists() and (data_dir / KEYWORD_FILENAME).exists():
            return candidate
    raise FileNotFoundError(
        f"Could not find data/{MAIN_DATA_FILENAME} and data/{KEYWORD_FILENAME}; confirm that the working directory is inside the project."
    )


PROJECT_ROOT = find_project_root(Path.cwd())
DATA_PATH = PROJECT_ROOT / "data" / MAIN_DATA_FILENAME
KEYWORD_PATH = PROJECT_ROOT / "data" / KEYWORD_FILENAME
FIG_DIR = PROJECT_ROOT / "output" / "figures"
TABLE_DIR = PROJECT_ROOT / "output" / "tables"
LOG_DIR = PROJECT_ROOT / "output" / "logs"

FIG_BASENAME = FIG_DIR / "Fig7_data_method_performance_sankey"
LINK_TABLE_PATH = TABLE_DIR / "08_sankey_links.csv"
LOG_PATH = LOG_DIR / "08_sankey_text.md"

print(f"Project root: {PROJECT_ROOT}")
print(f"Input data: {DATA_PATH.relative_to(PROJECT_ROOT)}")
print(f"Keyword table: {KEYWORD_PATH.relative_to(PROJECT_ROOT)}")


In [ ]:
# ===== 2. Read New Master Data and Keyword Table, Then Check Required Fields =====
# The new master table is the only primary data source for Fig. 7; the 37-keyword table is the controlled vocabulary for matched_* fields and category mapping.
df = pd.read_csv(DATA_PATH)
keyword_df = pd.read_csv(KEYWORD_PATH)

REQUIRED_COLUMNS = [
    "award_id",
    "project_title",
    "matched_keywords",
    "matched_method_terms",
    "matched_object_terms",
    "matched_performance_terms",
]
missing_columns = [col for col in REQUIRED_COLUMNS if col not in df.columns]
if missing_columns:
    raise KeyError(f"New master data is missing required columns: {missing_columns}")

if len(df) != EXPECTED_RECORDS_TOTAL:
    raise ValueError(f"New master data should contain {EXPECTED_RECORDS_TOTAL} records; got {len(df)}")

REQUIRED_KEYWORD_COLUMNS = ["term", "term_group"]
missing_keyword_columns = [col for col in REQUIRED_KEYWORD_COLUMNS if col not in keyword_df.columns]
if missing_keyword_columns:
    raise KeyError(f"Keyword table is missing required columns: {missing_keyword_columns}")

keyword_df = keyword_df[REQUIRED_KEYWORD_COLUMNS].copy()
keyword_df["term"] = keyword_df["term"].astype(str).str.strip()
keyword_df["term_group"] = keyword_df["term_group"].astype(str).str.strip()
keyword_df = keyword_df[keyword_df["term"].ne("")]

KEYWORD_GROUP_ORDER = ["method", "object_built_environment", "performance_environment"]
unexpected_groups = sorted(set(keyword_df["term_group"]) - set(KEYWORD_GROUP_ORDER))
if unexpected_groups:
    raise ValueError(f"Keyword table contains unknown term_group values: {unexpected_groups}")
if keyword_df["term"].duplicated().any():
    duplicated_terms = keyword_df.loc[keyword_df["term"].duplicated(), "term"].tolist()
    raise ValueError(f"Keyword table contains duplicated term values: {duplicated_terms}")

KEYWORDS_BY_GROUP = {
    group: keyword_df.loc[keyword_df["term_group"].eq(group), "term"].tolist()
    for group in KEYWORD_GROUP_ORDER
}

TEXT_COLUMN_CANDIDATES = [
    "project_title",
    "abstract_text",
    "keywords_raw",
    "outcomes_text",
    "project_keywords",
    "project_abstract_cn",
    "project_abstract_en",
    "conclusion_abstract",
]
TEXT_COLUMNS = [col for col in TEXT_COLUMN_CANDIDATES if col in df.columns]
if not TEXT_COLUMNS:
    raise KeyError("New master data lacks title/abstract/keyword/outcome text fields for manual traceability")

print(f"Records loaded: {len(df)}")
print(f"Keyword terms loaded: {len(keyword_df)}")
print(keyword_df["term_group"].value_counts().reindex(KEYWORD_GROUP_ORDER).to_string())
preview_columns = ["award_id", "project_title", "matched_keywords", "matched_method_terms", "matched_object_terms", "matched_performance_terms"]
print(df[preview_columns].head(3))


In [ ]:
# ===== 3. Parsing Functions and Category Dictionaries =====
# split_terms parses matched_method_terms, matched_object_terms, matched_performance_terms, and matched_keywords from the new master table.
# Category dictionaries map only controlled terms from the 37-keyword table; the denominator is fixed to the new master table N=9222.
TERM_SPLIT_RE = re.compile(r"[;；|,，、\s]+")


def split_terms(value) -> list[str]:
    """Parse matched terms separated by semicolons, enumeration commas, commas, or whitespace into a stripped list."""
    if pd.isna(value):
        return []
    return [term.strip() for term in TERM_SPLIT_RE.split(str(value)) if term.strip()]


def unique_preserve_order(values: list[str]) -> list[str]:
    """Deduplicate while preserving original order to retain the audit order from the keyword table and matched fields."""
    return list(dict.fromkeys(values))


def make_text_blob(row: pd.Series) -> str:
    """Combine text fields available for manual traceability; classification itself uses matched_* controlled terms."""
    return " ".join("" if pd.isna(row[col]) else str(row[col]) for col in TEXT_COLUMNS)


def make_signal_blob(*term_lists: list[str]) -> str:
    """Combine matched_* controlled-term signals as the only automatic input for category detection."""
    terms = []
    for term_list in term_lists:
        terms.extend(term_list)
    return " ".join(unique_preserve_order(terms))


# Data-source categories are identified jointly from method/object/keyword hits; object_built_environment terms mainly map to spatial/built-environment data layers.
SOURCE_PATTERNS = {
    "Remote-sensing imagery": ["遥感", "高分", "夜间灯光"],
    "Geospatial / BE layers": ["GIS", "地理信息", "规划", "城市群", "土地利用", "建筑", "绿地", "街道", "街区", "建成环境", "海绵", "城市形态"],
    "Multisource big data": ["大数据", "多源数据"],
    "Mobility and location traces": ["轨迹", "手机信令", "LBS"],
    "Street-level imagery": ["街景"],
    "3D / LiDAR data": ["三维", "LiDAR"],
    "POI and web-map data": ["POI"],
}

# Method-family categories use only controlled terms with term_group=method in the keyword table.
METHOD_PATTERNS = {
    "GIS and spatial analysis": ["GIS", "地理信息"],
    "Remote-sensing retrieval": ["遥感", "高分", "夜间灯光"],
    "AI / machine learning": ["人工智能", "机器学习", "深度学习"],
    "Data fusion and big-data analytics": ["多源数据", "大数据"],
    "Mobility / network analytics": ["手机信令", "LBS", "轨迹", "POI"],
    "Street-view visual analytics": ["街景"],
    "3D morphological modeling": ["三维", "LiDAR"],
}

# Environmental-performance categories: records enter the Sankey path only when matched_performance_terms map to the categories below.
PERFORMANCE_PATTERNS = {
    "Carbon and energy": ["碳", "能源"],
    "Thermal climate": ["气候", "热岛", "热环境"],
    "Ecological quality": ["生态环境"],
    "Resilience and flooding": ["韧性", "洪涝"],
    "Air pollution and exposure": ["暴露", "空气污染"],
}


def pattern_terms(patterns: dict[str, list[str]]) -> set[str]:
    """Extract all controlled terms covered by a category dictionary."""
    return {term for terms in patterns.values() for term in terms}


def assert_keyword_coverage(group: str, patterns: dict[str, list[str]]):
    """Check whether all controlled terms in a keyword-table group are included in the corresponding category mapping."""
    missing_terms = sorted(set(KEYWORDS_BY_GROUP[group]) - pattern_terms(patterns))
    if missing_terms:
        raise ValueError(f"{group} keywords are not yet mapped to the category dictionary: {missing_terms}")


assert_keyword_coverage("method", METHOD_PATTERNS)
assert_keyword_coverage("object_built_environment", SOURCE_PATTERNS)
assert_keyword_coverage("performance_environment", PERFORMANCE_PATTERNS)


def detect_categories(patterns: dict[str, list[str]], signal_blob: str) -> list[str]:
    """Detect categories in dictionary order and return deduplicated English category labels for direct use in figures and tables."""
    hits = []
    for label, keywords in patterns.items():
        if any(keyword and keyword in signal_blob for keyword in keywords):
            hits.append(label)
    return hits


def terms_without_category(terms: list[str], patterns: dict[str, list[str]]) -> list[str]:
    """List controlled terms that could not be mapped to categories for exclusion-reason auditing."""
    covered_terms = pattern_terms(patterns)
    return [term for term in terms if term not in covered_terms]


print("Configured categories from the controlled keyword table:")
print(f"  Data sources: {len(SOURCE_PATTERNS)}")
print(f"  Method families: {len(METHOD_PATTERNS)}")
print(f"  Performance outcomes: {len(PERFORMANCE_PATTERNS)}")


In [ ]:
# ===== 4. Classify Each Record and Retain Parsing Audit Information =====
# Multilabel records are not expanded into a Cartesian product here; first store each record's category sets.
# Only records with explicit, mappable environmental-performance controlled terms enter the Sankey path.
classified_rows = []
term_audit = []
path_exclusion_counter = Counter()

for idx, row in df.iterrows():
    method_terms = split_terms(row["matched_method_terms"])
    object_terms = split_terms(row["matched_object_terms"])
    performance_terms = split_terms(row["matched_performance_terms"])
    keyword_terms = split_terms(row["matched_keywords"])

    source_signal = make_signal_blob(method_terms, object_terms, keyword_terms)
    method_signal = make_signal_blob(method_terms, keyword_terms)
    performance_signal = make_signal_blob(performance_terms)

    data_sources = detect_categories(SOURCE_PATTERNS, source_signal) or ["Other data sources"]
    method_families = detect_categories(METHOD_PATTERNS, method_signal) or ["Other methods"]
    performance_outcomes = detect_categories(PERFORMANCE_PATTERNS, performance_signal)
    unmapped_performance_terms = terms_without_category(performance_terms, PERFORMANCE_PATTERNS)

    if performance_outcomes:
        path_exclusion_reason = ""
    elif not performance_terms:
        path_exclusion_reason = "no_matched_performance_terms"
    else:
        path_exclusion_reason = "unmapped_matched_performance_terms"
    if path_exclusion_reason:
        path_exclusion_counter[path_exclusion_reason] += 1

    term_audit.append({
        "award_id": row["award_id"],
        "method_terms_n": len(method_terms),
        "object_terms_n": len(object_terms),
        "performance_terms_n": len(performance_terms),
        "keyword_terms_n": len(keyword_terms),
        "data_sources_n": len(data_sources),
        "method_families_n": len(method_families),
        "performance_outcomes_n": len(performance_outcomes),
        "path_included": bool(performance_outcomes),
        "path_exclusion_reason": path_exclusion_reason,
        "unmapped_performance_terms": ";".join(unmapped_performance_terms),
    })

    if not performance_outcomes:
        continue

    classified_rows.append({
        "record_index": idx,
        "award_id": row["award_id"],
        "project_title": row["project_title"],
        "data_sources": data_sources,
        "method_families": method_families,
        "performance_outcomes": performance_outcomes,
        "matched_method_terms": method_terms,
        "matched_object_terms": object_terms,
        "matched_performance_terms": performance_terms,
        "matched_keywords": keyword_terms,
    })

classified_df = pd.DataFrame(classified_rows)
term_audit_df = pd.DataFrame(term_audit)
path_exclusion_df = (
    pd.DataFrame([{"reason": reason, "records": count} for reason, count in path_exclusion_counter.items()])
    .sort_values(["records", "reason"], ascending=[False, True])
    .reset_index(drop=True)
)

included_records = len(classified_df)
excluded_total = len(df) - included_records
excluded_no_performance = path_exclusion_counter.get("no_matched_performance_terms", 0)
excluded_unmapped_performance = path_exclusion_counter.get("unmapped_matched_performance_terms", 0)

if included_records + excluded_total != len(df):
    raise RuntimeError("Included and excluded path counts do not add back to the master table total")

print(f"Records with explicit environmental-performance pathway: {included_records}/{len(df)}")
print(f"Records excluded from Sankey path subset: {excluded_total}")
print(path_exclusion_df.to_string(index=False))
print(term_audit_df.describe(include="all").round(2))


In [ ]:
# ===== 5. Fractional Weighting, Category Merging, and Path Statistics =====
# Multilabel records use fractional weighting:
#   source -> method edge: each record contributes a total of 1 across all source-by-method combinations.
#   method -> performance edge: each record contributes a total of 1 across all method-by-performance combinations.
#   source -> method -> performance path: each record contributes a total of 1 across all three-way combinations.
# This preserves multilabel information while preventing multilabel records from having greater total weight than single-label records.
if classified_df.empty:
    raise ValueError("No records entered the Sankey pathway analysis; check matched_performance_terms and the category mapping.")

TOP_SOURCE_N = 5
TOP_METHOD_N = 6
TOP_PERFORMANCE_N = 6
OTHER_SOURCE = "Other data sources"
OTHER_METHOD = "Other methods"
OTHER_PERFORMANCE = "Other outcomes"


def fractional_category_counts(series_of_lists: pd.Series) -> Counter:
    """Compute category frequencies with fractional weighting by the number of categories within each record."""
    counter = Counter()
    for labels in series_of_lists:
        labels = list(dict.fromkeys(labels))
        for label in labels:
            counter[label] += 1 / len(labels)
    return counter


source_counts_raw = fractional_category_counts(classified_df["data_sources"])
method_counts_raw = fractional_category_counts(classified_df["method_families"])
performance_counts_raw = fractional_category_counts(classified_df["performance_outcomes"])

keep_sources = {label for label, _ in source_counts_raw.most_common(TOP_SOURCE_N)}
keep_methods = {label for label, _ in method_counts_raw.most_common(TOP_METHOD_N)}
keep_performances = {label for label, _ in performance_counts_raw.most_common(TOP_PERFORMANCE_N)}


def merge_labels(labels: list[str], keep: set[str], other_label: str) -> list[str]:
    """Merge low-frequency categories into Other and deduplicate again after merging."""
    merged = [label if label in keep else other_label for label in labels]
    return list(dict.fromkeys(merged))


edge_counter = Counter()
path_counter = Counter()
node_counter = {"Data source": Counter(), "Method family": Counter(), "Environmental performance": Counter()}

for _, row in classified_df.iterrows():
    sources = merge_labels(row["data_sources"], keep_sources, OTHER_SOURCE)
    methods = merge_labels(row["method_families"], keep_methods, OTHER_METHOD)
    performances = merge_labels(row["performance_outcomes"], keep_performances, OTHER_PERFORMANCE)

    # Node totals also use within-record fractional weighting so the three columns keep the same total flow.
    for source in sources:
        node_counter["Data source"][source] += 1 / len(sources)
    for method in methods:
        node_counter["Method family"][method] += 1 / len(methods)
    for performance in performances:
        node_counter["Environmental performance"][performance] += 1 / len(performances)

    for source in sources:
        for method in methods:
            edge_counter[("Data source", "Method family", source, method)] += 1 / (len(sources) * len(methods))

    for method in methods:
        for performance in performances:
            edge_counter[("Method family", "Environmental performance", method, performance)] += 1 / (len(methods) * len(performances))

    for source in sources:
        for method in methods:
            for performance in performances:
                path_counter[(source, method, performance)] += 1 / (len(sources) * len(methods) * len(performances))

source_order = [label for label, _ in node_counter["Data source"].most_common()]
method_order = [label for label, _ in node_counter["Method family"].most_common()]
performance_order = [label for label, _ in node_counter["Environmental performance"].most_common()]

print("Node totals after Other merge:")
for layer_name, order in [
    ("Data source", source_order),
    ("Method family", method_order),
    ("Environmental performance", performance_order),
]:
    print(f"\n{layer_name}")
    for label in order:
        print(f"  {label}: {node_counter[layer_name][label]:.2f}")

print("\nTop pathways:")
for (source, method, performance), value in path_counter.most_common(10):
    print(f"{value:.2f} | {source} -> {method} -> {performance}")


In [ ]:
# ===== 6. Export the Links Table =====
# The table stores the two merged edge segments: Data source -> Method family and Method family -> Environmental performance.
# value is the fractionally weighted record count; pct_of_included_records is the percentage of records included in pathway analysis.
link_rows = []
for (source_layer, target_layer, source, target), value in edge_counter.items():
    source_total = node_counter[source_layer][source]
    target_total = node_counter[target_layer][target]
    link_rows.append({
        "link_layer": f"{source_layer} -> {target_layer}",
        "source_layer": source_layer,
        "target_layer": target_layer,
        "source": source,
        "target": target,
        "value": value,
        "pct_of_included_records": value / included_records * 100,
        "source_node_total": source_total,
        "target_node_total": target_total,
    })

links_df = pd.DataFrame(link_rows)
links_df = links_df.sort_values(["link_layer", "value", "source", "target"], ascending=[True, False, True, True])
links_df["rank_within_layer"] = links_df.groupby("link_layer")["value"].rank(method="first", ascending=False).astype(int)

# Round numeric columns to 6 decimals so the CSV is stable and readable.
value_cols = ["value", "pct_of_included_records", "source_node_total", "target_node_total"]
links_df[value_cols] = links_df[value_cols].round(6)
TABLE_DIR.mkdir(parents=True, exist_ok=True)
links_df.to_csv(LINK_TABLE_PATH, index=False, encoding="utf-8-sig")

print(f"Links exported: {LINK_TABLE_PATH.relative_to(PROJECT_ROOT)}")
print(f"Link rows: {len(links_df)}")
links_df.head(12)


In [ ]:
# ===== 7. Custom Bezier Ribbon Plotting Functions =====
# Matplotlib does not provide a sufficiently stable and controllable three-column alluvial API, so PathPatch is used here to draw variable-width Bezier ribbons.
# All figure text is English so the figure can be used directly in an English manuscript.
LAYER_X = {
    "Data source": 0.08,
    "Method family": 0.50,
    "Environmental performance": 0.92,
}
NODE_WIDTH = 0.030
NODE_GAP = 0.018
Y_BOTTOM = 0.105
Y_TOP = 0.835

SOURCE_COLORS = {
    "Geospatial / BE layers": "#4B7F8C",
    "Remote-sensing imagery": "#80A7C8",
    "Multisource big data": "#8FAE82",
    "3D / LiDAR data": "#C9A66B",
    "Mobility and location traces": "#B9898D",
    "POI and web-map data": "#B7A36A",
    "Street-level imagery": "#AA8FA8",
    "Other data sources": "#B8B8B8",
}
METHOD_COLORS = {
    "Remote-sensing retrieval": "#6F9BBF",
    "GIS and spatial analysis": "#4F8F82",
    "Data fusion and big-data analytics": "#87A96B",
    "AI / machine learning": "#C7895B",
    "3D morphological modeling": "#B9A35B",
    "Mobility / network analytics": "#B9898D",
    "Street-view visual analytics": "#AA8FA8",
    "Other methods": "#B8B8B8",
}
PERFORMANCE_COLORS = {
    "Ecological quality": "#6EA672",
    "Resilience and flooding": "#6799C8",
    "Thermal climate": "#D39567",
    "Air pollution and exposure": "#9E8AC7",
    "Carbon and energy": "#C66C5A",
    "Other outcomes": "#B8B8B8",
}

DISPLAY_LABELS = {
    "Geospatial / BE layers": "Geospatial and\nbuilt-environment layers",
    "Remote-sensing imagery": "Remote-sensing\nimagery",
    "Multisource big data": "Multisource\nbig data",
    "3D / LiDAR data": "3D / LiDAR\ndata",
    "Mobility and location traces": "Mobility and\nlocation traces",
    "POI and web-map data": "POI and\nweb-map data",
    "Street-level imagery": "Street-level\nimagery",
    "Other data sources": "Other data\nsources",
    "Remote-sensing retrieval": "Remote-sensing\nretrieval",
    "GIS and spatial analysis": "GIS and spatial\nanalysis",
    "Data fusion and big-data analytics": "Data fusion and\nbig-data analytics",
    "AI / machine learning": "AI / machine\nlearning",
    "3D morphological modeling": "3D morphological\nmodeling",
    "Mobility / network analytics": "Mobility / network\nanalytics",
    "Street-view visual analytics": "Street-view visual\nanalytics",
    "Other methods": "Other\nmethods",
    "Ecological quality": "Ecological\nquality",
    "Resilience and flooding": "Resilience and\nflooding",
    "Thermal climate": "Thermal\nclimate",
    "Air pollution and exposure": "Air pollution\nand exposure",
    "Carbon and energy": "Carbon and\nenergy",
    "Other outcomes": "Other\noutcomes",
}


def lighten(color: str, amount: float = 0.60) -> tuple[float, float, float]:
    """Blend a color toward white to produce a low-saturation node fill."""
    rgb = np.array(mcolors.to_rgb(color))
    return tuple(rgb + (1 - rgb) * amount)


def layer_layout(order: list[str], totals: Counter, total_flow: float, scale_y: float) -> dict[str, dict[str, float]]:
    """Compute each layer's node y0/y1/center values from node totals, a shared scale, and fixed spacing."""
    total_height = total_flow * scale_y + NODE_GAP * (len(order) - 1)
    current_top = 0.5 + total_height / 2
    layout = {}
    for label in order:
        height = totals[label] * scale_y
        y1 = current_top
        y0 = current_top - height
        layout[label] = {"y0": y0, "y1": y1, "yc": (y0 + y1) / 2, "height": height}
        current_top = y0 - NODE_GAP
    return layout


def cubic_ribbon_path(x0: float, y0_center: float, x1: float, y1_center: float, width: float) -> MplPath:
    """Generate a ribbon whose upper and lower boundaries are cubic Bezier curves."""
    half_width = width / 2
    dx = x1 - x0
    c = dx * 0.42
    verts = [
        (x0, y0_center + half_width),
        (x0 + c, y0_center + half_width),
        (x1 - c, y1_center + half_width),
        (x1, y1_center + half_width),
        (x1, y1_center - half_width),
        (x1 - c, y1_center - half_width),
        (x0 + c, y0_center - half_width),
        (x0, y0_center - half_width),
        (x0, y0_center + half_width),
    ]
    codes = [
        MplPath.MOVETO,
        MplPath.CURVE4,
        MplPath.CURVE4,
        MplPath.CURVE4,
        MplPath.LINETO,
        MplPath.CURVE4,
        MplPath.CURVE4,
        MplPath.CURVE4,
        MplPath.CLOSEPOLY,
    ]
    return MplPath(verts, codes)


def stack_positions_for_edges(edge_values: dict[tuple[str, str], float], left_order: list[str], right_order: list[str], left_layout, right_layout, scale_y: float):
    """Compute stacked center points at both ends of a two-column edge segment so ribbons are segmented inside nodes by target/source order."""
    left_cursor = {label: left_layout[label]["y1"] for label in left_order}
    right_cursor = {label: right_layout[label]["y1"] for label in right_order}
    positions = {}

    for left in left_order:
        for right in right_order:
            value = edge_values.get((left, right), 0)
            if value <= 0:
                continue
            height = value * scale_y
            y0_center = left_cursor[left] - height / 2
            left_cursor[left] -= height
            positions[(left, right)] = {"left_y": y0_center, "height": height, "value": value}

    for right in right_order:
        for left in left_order:
            value = edge_values.get((left, right), 0)
            if value <= 0:
                continue
            height = value * scale_y
            y1_center = right_cursor[right] - height / 2
            right_cursor[right] -= height
            positions[(left, right)]["right_y"] = y1_center

    return positions


def save_publication_figure(fig, basename: Path):
    """Export SVG/PDF/TIFF/PNG in the four formats required by the task."""
    basename.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(f"{basename}.svg", bbox_inches="tight", facecolor="white")
    fig.savefig(f"{basename}.pdf", bbox_inches="tight", facecolor="white")
    fig.savefig(f"{basename}.png", dpi=300, bbox_inches="tight", facecolor="white")
    fig.savefig(f"{basename}.tiff", dpi=600, bbox_inches="tight", facecolor="white", pil_kwargs={"compression": "tiff_lzw"})


In [ ]:
# ===== 8. Draw Fig. 7 and Export Four Formats =====
# The three columns should have nearly identical total flow; use one shared y-scale so widths are comparable.
total_flow = included_records
max_nodes_per_layer = max(len(source_order), len(method_order), len(performance_order))
scale_y = (Y_TOP - Y_BOTTOM - NODE_GAP * (max_nodes_per_layer - 1)) / total_flow

source_layout = layer_layout(source_order, node_counter["Data source"], total_flow, scale_y)
method_layout = layer_layout(method_order, node_counter["Method family"], total_flow, scale_y)
performance_layout = layer_layout(performance_order, node_counter["Environmental performance"], total_flow, scale_y)

edge_source_method = {(source, target): value for (src_layer, tgt_layer, source, target), value in edge_counter.items() if src_layer == "Data source"}
edge_method_performance = {(source, target): value for (src_layer, tgt_layer, source, target), value in edge_counter.items() if src_layer == "Method family"}

sm_positions = stack_positions_for_edges(edge_source_method, source_order, method_order, source_layout, method_layout, scale_y)
mp_positions = stack_positions_for_edges(edge_method_performance, method_order, performance_order, method_layout, performance_layout, scale_y)

fig, ax = plt.subplots(figsize=(9.1, 6.4))
fig.patch.set_facecolor("white")
ax.set_facecolor("white")
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.axis("off")

# Draw ribbons first, stacking from small to large so thick ribbons remain visible.
for (source, method), pos in sorted(sm_positions.items(), key=lambda item: item[1]["value"]):
    x0 = LAYER_X["Data source"] + NODE_WIDTH / 2
    x1 = LAYER_X["Method family"] - NODE_WIDTH / 2
    color = SOURCE_COLORS.get(source, "#B8B8B8")
    patch = PathPatch(
        cubic_ribbon_path(x0, pos["left_y"], x1, pos["right_y"], pos["height"]),
        facecolor=color,
        edgecolor="none",
        alpha=0.34,
        zorder=1,
    )
    ax.add_patch(patch)

for (method, performance), pos in sorted(mp_positions.items(), key=lambda item: item[1]["value"]):
    x0 = LAYER_X["Method family"] + NODE_WIDTH / 2
    x1 = LAYER_X["Environmental performance"] - NODE_WIDTH / 2
    color = METHOD_COLORS.get(method, "#B8B8B8")
    patch = PathPatch(
        cubic_ribbon_path(x0, pos["left_y"], x1, pos["right_y"], pos["height"]),
        facecolor=color,
        edgecolor="none",
        alpha=0.34,
        zorder=1,
    )
    ax.add_patch(patch)

# Then draw nodes and English labels. Labels show fractionally weighted record counts so reviewers can trace the flow meaning.
def draw_nodes(layer_name: str, order: list[str], layout: dict[str, dict[str, float]], color_map: dict[str, str], label_side: str):
    x = LAYER_X[layer_name]
    for label in order:
        y0, y1 = layout[label]["y0"], layout[label]["y1"]
        base_color = color_map.get(label, "#B8B8B8")
        rect = Rectangle(
            (x - NODE_WIDTH / 2, y0),
            NODE_WIDTH,
            y1 - y0,
            facecolor=lighten(base_color, 0.42),
            edgecolor=base_color,
            linewidth=0.8,
            zorder=3,
        )
        ax.add_patch(rect)

        label_text = f"{DISPLAY_LABELS.get(label, label)}\n{node_counter[layer_name][label]:.1f}"
        if label_side == "left":
            ax.text(x - NODE_WIDTH / 2 - 0.012, (y0 + y1) / 2, label_text, ha="right", va="center", fontsize=7.2, color="#222222", linespacing=1.08)
        elif label_side == "right":
            ax.text(x + NODE_WIDTH / 2 + 0.012, (y0 + y1) / 2, label_text, ha="left", va="center", fontsize=7.2, color="#222222", linespacing=1.08)
        else:
            ax.text(x, (y0 + y1) / 2, label_text, ha="center", va="center", fontsize=6.8, color="#222222", linespacing=1.05)


draw_nodes("Data source", source_order, source_layout, SOURCE_COLORS, "left")
draw_nodes("Method family", method_order, method_layout, METHOD_COLORS, "center")
draw_nodes("Environmental performance", performance_order, performance_layout, PERFORMANCE_COLORS, "right")

# Keep the three column structure labels; do not add an overall title or count note at the top of the figure.
for layer_name in ["Data source", "Method family", "Environmental performance"]:
    ax.text(
        LAYER_X[layer_name],
        0.885,
        layer_name,
        ha="center",
        va="center",
        fontsize=10.5,
        fontweight="bold",
        color="#1f1f1f",
    )


save_publication_figure(fig, FIG_BASENAME)
plt.show()

print("Exported figure files:")
for suffix in ["svg", "pdf", "tiff", "png"]:
    out_path = Path(f"{FIG_BASENAME}.{suffix}")
    print(f"  {out_path.relative_to(PROJECT_ROOT)} | {out_path.stat().st_size:,} bytes")


In [ ]:
# ===== 9. Write the Text Log =====
# The log provides traceable numbers for the manuscript Results/Methods and is not figure text.
node_count = len(source_order) + len(method_order) + len(performance_order)
edge_count = len(links_df)
top_paths = path_counter.most_common(10)

node_lines = []
for layer_name, order in [
    ("Data source", source_order),
    ("Method family", method_order),
    ("Environmental performance", performance_order),
]:
    node_lines.append(f"### {layer_name}")
    for label in order:
        node_lines.append(f"- {label}: {node_counter[layer_name][label]:.2f}")
    node_lines.append("")

top_path_lines = [
    f"{rank}. {source} -> {method} -> {performance}: {value:.2f}"
    for rank, ((source, method, performance), value) in enumerate(top_paths, start=1)
]

if path_exclusion_df.empty:
    exclusion_lines = ["- No excluded records"]
else:
    exclusion_lines = [f"- {row.reason}: {int(row.records)}" for row in path_exclusion_df.itertuples(index=False)]

log_text = f"""# 08 Data-Method-Performance Pathway Log

Generated at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}

## Inputs and Scope
- Input data: `data/{MAIN_DATA_FILENAME}`
- Keyword table: `data/{KEYWORD_FILENAME}`
- Master-table records: {len(df)}
- Records included in pathway analysis: {included_records}
- Records excluded from the Sankey pathway: {excluded_total}
{chr(10).join(exclusion_lines)}
- Path inclusion rule: `matched_performance_terms` must hit at least one mappable environmental-performance category.
- Parsed fields: `matched_method_terms`, `matched_object_terms`, `matched_performance_terms`, and `matched_keywords`.
- Weighting scope: multilabel records use fractional weighting; each included record contributes a total of 1 to each edge segment.
- Merge scope: Data source keeps Top {TOP_SOURCE_N}, Method family keeps Top {TOP_METHOD_N}, and Environmental performance keeps Top {TOP_PERFORMANCE_N}; low-frequency categories are merged into Other.

## Figure Scale
- Node count: {node_count}
- Edge count: {edge_count}
- Data source nodes: {len(source_order)}
- Method family nodes: {len(method_order)}
- Environmental performance nodes: {len(performance_order)}

## Node Weights
{chr(10).join(node_lines)}
## Top 10 Pathways
{chr(10).join(top_path_lines)}

## Output Files
- `output/figures/Fig7_data_method_performance_sankey.svg`
- `output/figures/Fig7_data_method_performance_sankey.pdf`
- `output/figures/Fig7_data_method_performance_sankey.tiff`
- `output/figures/Fig7_data_method_performance_sankey.png`
- `output/tables/08_sankey_links.csv`
"""

LOG_DIR.mkdir(parents=True, exist_ok=True)
LOG_PATH.write_text(log_text, encoding="utf-8")
print(f"Log exported: {LOG_PATH.relative_to(PROJECT_ROOT)}")
print(log_text[:1400])


In [ ]:
# ===== 10. Summary Check =====
# This cell provides a quick view of final node/edge counts and top pathways for the coordinating subagent summary.
summary_top_paths = path_counter.most_common(10)
summary_node_count = len(source_order) + len(method_order) + len(performance_order)
summary_edge_count = len(links_df) if "links_df" in globals() else len(edge_counter)
summary = {
    "input_data": f"data/{MAIN_DATA_FILENAME}",
    "keyword_table": f"data/{KEYWORD_FILENAME}",
    "records_total": len(df),
    "records_included": included_records,
    "records_excluded_total": excluded_total,
    "records_excluded_no_matched_performance_terms": excluded_no_performance,
    "records_excluded_unmapped_performance_terms": excluded_unmapped_performance,
    "path_subset_rule": "matched_performance_terms maps to at least one environmental-performance category",
    "node_count": summary_node_count,
    "edge_count": summary_edge_count,
    "top_path": " -> ".join(summary_top_paths[0][0]) if summary_top_paths else None,
    "top_path_value": round(summary_top_paths[0][1], 2) if summary_top_paths else None,
}
summary
